# Qué le hicieron los modelos entrenados a su representación (v1)

El informe dice qué método gana. No dice si ganó **alineando los dominios clase
por clase**, que es lo que las dos formulaciones afirman. Esa afirmación vive en
la representación, así que se mide ahí.

Este cuaderno no entrena nada. Carga los checkpoints que guardó la campaña y lee
qué representan — que es lo que hace barata cualquier pregunta posterior: todo lo
que se pueda calcular a partir de pesos y datos se puede agregar cuando se le
ocurra a alguien, sin volver a correr la campaña.

Dos reglas gobiernan cada figura de abajo, y las dos existen para que un dibujo no
salga más lindo que la corrida que describe.

**Una sola semilla de exhibición, no la mediana de cada método.** Paneles sacados
de semillas distintas se diferenciarían en el método *y* en el sorteo, y como la
semilla fija la partición, ni siquiera compartirían las bolsas. La semilla se
elige con una regla que no favorece a nadie: aquella cuya exactitud media
**sobre todos los métodos** es la mediana.

**Artefactos medianos, nunca el mejor.** El mejor de N crece con la dispersión
propia del método, así que comparar mejor contra mejor le hace el regalo más
grande al método más ruidoso.

> **De dónde sale el registro que lee este cuaderno.**
>
> `summary.json` y `runs.jsonl` los escribió `tools/bridge.py` mezclando los diez
> shards que volvieron de los envíos remotos — 30 semillas, 20 épocas, las diez en
> la misma tarjeta. No los escribió `Benchmark_Campaign_v1.ipynb`, que no se
> re-ejecuta y cuyas salidas son de una corrida local anterior. La provenance del
> registro está en `Results/Benchmark/PROVENANCE.txt` y se muestra más abajo.


In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [ ]:
# The repository was resolved by the cell above -- the forge's own, travelling
# byte for byte -- and this one only uses it. `REPOSITORY` is still the name
# every cell below reads, so nothing below changes.
import os
import sys
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import json

import torch
from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, figures, harness, latent, tables


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo.

    Es la misma cadena que va al registro, renderizada. Nada se vuelve a calcular
    acá: si esto y el archivo dijeran cosas distintas, habría dos versiones del
    mismo número y ninguna forma de saber cuál se movió.
    """
    display(Markdown(text))


device = harness.resolve_device()
# La corrida vigente primero: decide de qué árbol salen los pesos, y sin eso
# el guarda miraba `Models/Benchmark` mientras el ensayo escribía en
# `Models/Pilot/Benchmark` --- y se negaba con checkpoints en disco.
from MIL_CREDA_Benchmark import contamination as _eje

_vigente = _eje.in_force(0.0, "campaign")
if _vigente is None:
    raise SystemExit("no hay registro para ρ=0: ni corrida completa ni ensayo.")
ES_ENSAYO = bool(_vigente["pilot"])
PROCEDENCIA = _eje.source_note(_vigente, 0.0)
if not latent.available(0.0, ES_ENSAYO):
    raise SystemExit(
        f"no hay checkpoints en "
        f"{config.models_for(0.0, 'campaign', ES_ENSAYO)}. "
        f"Corré primero la campaña: guarda "
        f"las repeticiones más cercanas a la mediana de cada celda."
    )

# Igual que el informe: la completa primero, el ensayo sólo si no hay otra, y
# dicho en voz alta de cuál se está leyendo.
from MIL_CREDA_Benchmark import contamination as _eje

_vigente = _eje.in_force(0.0, "campaign")
if _vigente is None:
    raise SystemExit("no hay registro para ρ=0: ni corrida completa ni ensayo.")
runs, summary = _vigente["runs"], _vigente["summary"]
ES_ENSAYO = bool(_vigente["pilot"])
PROCEDENCIA = _eje.source_note(_vigente, 0.0)

# Los checkpoints y el registro de al lado describen una misma corrida, o no se
# mide nada. `bound()` se niega cuando no coinciden, y esa es la única forma de
# que "usa los de la campaña" sea una garantía y no una coincidencia: un
# directorio con checkpoints válidos de una corrida anterior más chica carga,
# mide y se renderiza igual que los correctos, bajo el sello de este registro.
found = latent.bound(latent.available(0.0, ES_ENSAYO), summary)
seed = latent.display_seed(runs)
print(f"{len(found)} checkpoints en {device} · semilla de exhibición: {seed}")

In [ ]:
readings = [latent.analyse(record, device) for record in found]
print(f"{len(readings)} checkpoints medidos")

# El mismo análisis sobre la campaña contaminada, al nivel que el paquete
# declara. `RHO` está fijado antes de que nada corriera --- el punto medio del
# rango --- así que ningún resultado pudo haberlo elegido.
#
# `latent.available(RHO)` lee el árbol de pesos de esa tasa y no el limpio: los
# dos existen porque `models_for` los separa, y leer el equivocado mediría la
# corrida que no es sin que nada lo notara.
RHO = config.NOISE_REPORTED

# Por `in_force` y no por `results_for`: la mitad contaminada necesita el mismo
# respaldo que la limpia --- corrida completa primero, ensayo si no hay otra ---
# y leerla del árbol completo a secas la mandaba a un `summary.json` que no
# existe mientras el ensayo tenía el suyo dos directorios más allá.
_ruido = _eje.in_force(RHO, "campaign")
hallados_ruido = latent.available(RHO, bool(_ruido["pilot"])) if _ruido else []
if hallados_ruido:
    readings_ruido = [latent.analyse(r, device)
                      for r in latent.bound(hallados_ruido, _ruido["summary"])]
    print(f"{len(hallados_ruido)} checkpoints contaminados · "
          f"{_eje.source_note(_ruido, RHO)}")
else:
    readings_ruido = []
    print(f"sin checkpoints a ρ={RHO:g}: la campaña contaminada no dejó pesos")


## Bajo qué corrió todo esto

Los límites de la corrida se dicen una vez, acá, y atan todo lo que sigue. No se
repiten en cada tabla: un aviso que aparece ocho veces enseña a saltearlo, y la vez
que importaba era una de esas ocho.

In [ ]:
show(tables.stamp(summary["reduction"], markdown=True))

In [ ]:
show(PROCEDENCIA)


## 1 · ¿Son redundantes los dos pisos?

`Baseline` y `MIL-Baseline` no aprenden ninguna adaptación: los dos son el mismo
modelo sin término, uno midiendo sobre instancias y el otro sobre bolsas. Medimos
si su representación coincide —la razón entre distancias y la separabilidad de
dominio, transferencia por transferencia— para decidir si la grilla de abajo lleva
dos columnas de piso o una. Buscamos que la diferencia entre ambos sea menor que la
que separa a cualquiera de ellos de un método con adaptación: si lo es, la segunda
columna no aporta nada que la primera no muestre, y ocupa espacio que sí importa.

La decisión la toma la medición, no el gusto: el código compara y colapsa solo si
encontró que hay algo que colapsar.

In [ ]:
show(tables.objective("floors"))

In [ ]:
transfers = tables.best_transfers(runs)
# Por escala, como todo lo demás de este cuaderno: leía los pesos del árbol
# COMPLETO mientras la campaña de ensayo escribía en el suyo, así que contestaba
# «sin checkpoints de los dos pisos» con los checkpoints en disco.
pisos = latent.floors_agree(transfers, seed, device, rate=0.0, pilot=ES_ENSAYO)
# `floors_agree` responde `agree: None` y un motivo cuando no tiene
# checkpoint de los dos pisos. Leer `byTransfer` sin preguntar convierte
# esa ausencia medida en un error, que es lo contrario de reportarla.
if "byTransfer" not in pisos:
    print(f"  {pisos['detail']}")
for fila in pisos.get("byTransfer", []):
    print(f"  {fila['transfer']:<7} razón A={fila['ratio']['A']:.3f} "
          f"B={fila['ratio']['B']:.3f} (dif {fila['ratioGap']:.3f})   "
          f"separab. A={fila['separability']['A']:.3f} "
          f"B={fila['separability']['B']:.3f} (dif {fila['separabilityGap']:.3f})")

paneles = ([a for a in config.LATENT_PANELS if a != "B"] if pisos["agree"]
           else config.LATENT_PANELS)

In [ ]:
show(pisos["detail"])

## 2 · La grilla del espacio de representación

Cada panel es el espacio donde cada método dejó a sus datos, con las filas por
transferencia y las columnas por método. El color es la clase, los círculos son la
fuente y los triángulos el destino. Buscamos que los dos dominios se mezclen dentro
de cada clase y que las clases sigan separadas entre sí: mezclar todo también junta
los dominios, y eso no es alinear sino colapsar.

Cada panel se dibuja al nivel de instancia, entrene el método sobre instancias o
sobre bolsas, porque es el único espacio que todos tienen y el único modo de que
todos los paneles lleven la misma cantidad de puntos. Un panel más denso que otro se
lee como más cobertura, y sería la unidad estadística dibujada y no la alineación.
Todos los paneles de una fila miran los mismos sujetos.

In [ ]:
show(tables.objective("grid"))

In [ ]:
# Por escala, igual que la mitad contaminada de más abajo. Este cuaderno LEÍA
# los pesos por `ES_ENSAYO` y ESCRIBÍA siempre en el árbol completo, así que un
# análisis latente de ensayo se dibujaba encima del de la corrida completa.
DESTINO = config.results_for(0.0, "campaign", ES_ENSAYO)
grid_path = DESTINO / "latent" / "grid.pdf"
# El destino ya iba por escala; los pesos no. La grilla se dibujaba entera en
# gris ---un panel apagado por celda--- leyendo un árbol vacío.
display(figures.inline(latent.latent_grid(grid_path, paneles, transfers, seed,
                                          device, 0.0, ES_ENSAYO)))

## 3 · La razón entre distancias

Cuánto se acercaron los dominios sin que las clases se acercaran entre sí: la
distancia media entre los centroides de la misma clase en los dos dominios, dividida
por la distancia media entre clases distintas dentro de un dominio. Es el número que
pone cifras a lo que la grilla de arriba deja ver. Buscamos que baje, **pero solo
sirve leída junto con su denominador**: una razón puede caer porque los dominios se
acercaron o porque todo colapsó, y sola llama éxito a las dos cosas. Por eso las dos
distancias van por separado justo abajo.

In [ ]:
show(tables.objective("geometry.ratio"))

In [ ]:
show(tables.render_readings(readings, "geometry.ratio",
                            "razón: misma clase entre dominios / entre clases",
                            markdown=True))

In [ ]:
show(tables.conclusion_geometry(readings))

#### La razón entre distancias, con el material contaminado

La misma lectura sobre la campaña a ρ del paquete. La conclusión de abajo informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_readings_contaminated(
    readings_ruido, "geometry.ratio", "La razón entre distancias", RHO, markdown=True))


In [ ]:
show(tables.conclusion_readings_versus_clean(
    readings, readings_ruido, "geometry.ratio", RHO)
     if readings_ruido else
     "Sin la corrida contaminada no hay distancia que medir.")

### 3b · Las dos distancias por separado

Los dos numeradores de la razón anterior, cada uno en su tabla porque cada tabla es
una medición: primero la distancia de una misma clase **entre los dos dominios**,
que es lo que la adaptación tendría que estar acercando; después la distancia
**entre clases distintas dentro de un dominio**.

Van juntas y se concluyen juntas, y eso no es una comodidad de formato. La primera
bajando puede ser alineación o puede ser colapso, y sola llama éxito a las dos
cosas; lo que las distingue es cuánto bajó la segunda con ella.

**Qué se mide, exactamente.** Estas dos distancias no son la discrepancia local
$d_j^{2}$ de la Ec. (31): aquella vive en $\mathcal H_I$ y la calcula el método
como parte de su pérdida. Estas son **diagnósticos de esta verificación**,
euclidianas y sobre las mismas representaciones que el modelo entrenado produce,
definidas acá y en ninguna otra parte de la propuesta.

Sea $\mathcal C$ el conjunto de clases y $r$ la fila que cada arquitectura alinea:
para un brazo de unidad bolsa es la representación $\mathbf z_i \in \mathbb R^{d}$
de la Ec. (16), y para uno de unidad instancia el embedding
$\mathbf h_{i,a} \in \mathbb R^{d}$ de la Ec. (13). Sobre las bolsas de evaluación,
el centroide de la clase $c$ en el dominio $u \in \{s, t\}$ es

$$
\boldsymbol\gamma_{c}^{u}
=
\frac{1}{\lvert \mathcal I_{c}^{u} \rvert}
\sum_{i \in \mathcal I_{c}^{u}} r_i^{u},
\qquad
\mathcal I_{c}^{u} = \left\{ i : c_i^{u} = c \right\}.
$$

La primera tabla promedia, sobre las clases presentes en los dos dominios, la
distancia entre el mismo centroide de clase a un lado y al otro:

$$
D_{\mathrm{cruz}}
=
\frac{1}{\lvert \mathcal C_{\cap} \rvert}
\sum_{c \in \mathcal C_{\cap}}
\left\lVert \boldsymbol\gamma_{c}^{s} - \boldsymbol\gamma_{c}^{t} \right\rVert_{2},
\qquad
\mathcal C_{\cap} = \left\{ c : \mathcal I_{c}^{s} \neq \emptyset \ \wedge\ \mathcal I_{c}^{t} \neq \emptyset \right\}.
$$

La segunda promedia, sobre los dos dominios juntos, la distancia entre centroides
de clases distintas dentro de un mismo dominio:

$$
D_{\mathrm{entre}}
=
\frac{1}{\lvert \mathcal P \rvert}
\sum_{(u, c, c') \in \mathcal P}
\left\lVert \boldsymbol\gamma_{c}^{u} - \boldsymbol\gamma_{c'}^{u} \right\rVert_{2},
\qquad
\mathcal P = \left\{ (u, c, c') : u \in \{s, t\},\ c, c' \in \mathcal C^{u},\ c < c' \right\},
$$

con $\mathcal C^{u}$ las clases presentes en el dominio $u$ — cada dominio aporta
sus propios pares, y no solo los de $\mathcal C_{\cap}$.

La razón de la sección anterior es $D_{\mathrm{cruz}} / D_{\mathrm{entre}}$, y por
eso las dos se leen juntas: el mismo cociente baja porque el numerador se juntó o
porque el denominador se encogió, y son cosas opuestas.

**Contra qué se lee cada número.** Ninguno de los dos tiene escala fija —
$\mathcal F_{\theta}$ se asienta donde quiera en cada corrida — así que un valor
absoluto no dice nada y dos métodos no se comparan por sus valores crudos. Cada
brazo se lee **contra su propio piso, dentro de la misma transferencia**: el mismo
brazo con el término de adaptación apagado. Eso es lo que reporta la conclusión, en
cambio porcentual, y el criterio está declarado en el recuadro que sigue a este
párrafo.


In [ ]:
show(tables.render_readings(readings, "geometry.crossDomainSameClass",
                            "misma clase, entre los dos dominios", markdown=True))

#### La distancia entre dominios, con el material contaminado

La misma lectura sobre la campaña a ρ del paquete. La conclusión de abajo informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.render_readings_contaminated(
    readings_ruido, "geometry.crossDomainSameClass", "La distancia entre dominios", RHO, markdown=True))


In [ ]:
show(tables.conclusion_readings_versus_clean(
    readings, readings_ruido, "geometry.crossDomainSameClass", RHO)
     if readings_ruido else
     "Sin la corrida contaminada no hay distancia que medir.")

### 3c · Clases distintas dentro de un dominio

In [ ]:
show(tables.render_readings(readings, "geometry.betweenClasses",
                            "clases distintas, dentro de un dominio", markdown=True))

#### Clases distintas dentro de un dominio, contaminado

In [ ]:
show(tables.render_readings_contaminated(
    readings_ruido, "geometry.betweenClasses",
    "clases distintas, dentro de un dominio", RHO, markdown=True))

In [ ]:
show(tables.conclusion_readings_versus_clean(
    readings, readings_ruido, "geometry.betweenClasses", RHO)
     if readings_ruido else
     "Sin la corrida contaminada no hay distancia que medir.")

In [ ]:
show(tables.conclusion_distances(readings))

## 4 · Qué tan separables siguen los dominios

Con qué exactitud una regla lineal entrenada sobre la representación puede decir de
qué dominio vino cada punto. Es la lectura complementaria de la anterior: mide si
quedó información de dominio, que es justo lo que la adaptación tendría que haber
borrado. Buscamos que se acerque al azar de esa decisión —que son dos etiquetas, así
que 0.5— y no que baje todo lo posible: por debajo del azar la regla estaría
acertando al revés, que es otra forma de decir que la información sigue ahí.

In [ ]:
show(tables.objective("domainSeparability"))

In [ ]:
show(tables.render_readings(readings, "domainSeparability",
                            f"exactitud de un clasificador de dominio "
                            f"(azar {tables.DOMAIN_CHANCE:.3f})", markdown=True))

In [ ]:
show(tables.conclusion_separability(readings))

#### La separabilidad de los dominios, con el material contaminado

La misma lectura sobre la campaña a ρ del paquete. La conclusión de abajo informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_readings_contaminated(
    readings_ruido, "domainSeparability", "La separabilidad de los dominios", RHO, markdown=True))


In [ ]:
show(tables.conclusion_readings_versus_clean(
    readings, readings_ruido, "domainSeparability", RHO)
     if readings_ruido else
     "Sin la corrida contaminada no hay distancia que medir.")

## 5 · La correspondencia, sujeto por sujeto

Para una bolsa de destino de cada clase, dónde queda respecto de las bolsas de
fuente en el espacio de representación. Las filas son las mismas tres
transferencias y las columnas son el piso, el mismo método sin el término local, y
el completo: el peldaño donde ese término vive. **El color es la clase**, los
círculos son bolsas de fuente y los triángulos bolsas de destino, en los tres
paneles por igual. Buscamos que el triángulo destacado caiga entre círculos de su
propio color — eso es haber quedado cerca de su clase al otro lado del dominio.

**La línea aparece en una sola columna, y esa es la afirmación.** Solo `MIL-CREDA`
declara correspondencia por bolsas; en los otros dos no hay emparejamiento que
trazar, y lo que se dibujaría es la vecina más cercana, que es una consecuencia de
dónde quedaron los puntos y no algo que el método afirme. El sujeto destacado sí se
marca en los tres paneles, porque es la misma bolsa a lo largo de toda la fila y
seguirla de panel en panel es la comparación que la figura ofrece; lo que no se
traza es una conclusión que el método no computó. Los aciertos de los tres se
cuentan igual y van en la tabla de abajo.

Se destaca **la misma bolsa de cada clase en todos los paneles de una fila**, y es
la **mediana** de su clase por masa de correspondencia, nunca la mejor: la mejor
bolsa queda bien ubicada bajo cualquier método, incluido el piso que no aprendió
ninguna correspondencia, y una figura que no puede salir mal no está midiendo nada.
La vecina se calcula con el kernel de bolsas **en el espacio de representación** —
nunca euclídea, que el método no usa, y nunca en la proyección, que ilustraría a
UMAP.

In [ ]:
show(tables.objective("correspondence"))

In [ ]:
# `0.0, ES_ENSAYO`, igual que su gemela contaminada de más abajo: la mitad
# limpia era la única de las dos que no decía de qué corrida salen los pesos.
correspondencia = latent.correspondence_grid(
    DESTINO / "latent" / "correspondence.pdf", config.BAG_PANELS, transfers,
    seed, device, 0.0, ES_ENSAYO)
display(figures.inline(correspondencia["figure"]))

### 5b · Los aciertos y la masa, como números

Los mismos paneles de arriba contados: de cuántas clases la vecina de fuente resultó
ser de la clase correcta, y cuánta masa de correspondencia cae en la clase verdadera.
Vivían dentro de la figura, como pie de cada panel, donde no se comparaban de una
fila a otra y de paso hacían que las tres columnas se vieran igual de afirmativas.
Cada método va etiquetado por lo que de verdad computa, así que se ve de un vistazo
cuál es el único que declara el término local. Buscamos que ese sea el que más
acierta, y por encima del azar de elegir una clase.

In [ ]:
show(tables.objective("correspondence.massOnTrueClass"))

In [ ]:
show(tables.render_correspondence(correspondencia["scored"], markdown=True))

#### La correspondencia, con el material contaminado

Es la sección donde el término local vive, así que es donde la contaminación debería doler más: una bolsa con instancias de otra clase es exactamente lo que la correspondencia sujeto-a-sujeto tiene que sobrevivir.

In [ ]:
correspondencia_ruido = (latent.correspondence_grid(
    config.results_for(RHO, "campaign", ES_ENSAYO) / "latent" / "correspondence.pdf",
    config.BAG_PANELS, transfers, seed, device, RHO, ES_ENSAYO)
    if readings_ruido else None)
show(tables.render_correspondence(correspondencia_ruido["scored"], markdown=True)
     if correspondencia_ruido else
     f"Sin checkpoints contaminados a ρ={RHO:g} no hay segunda tabla. "
     f"No está vacía: no existe.")


In [ ]:
show(tables.conclusion_correspondence(correspondencia_ruido["scored"], readings_ruido)
     if correspondencia_ruido else
     "Sin checkpoints contaminados no hay segunda conclusión.")

In [ ]:
show(tables.conclusion_correspondence(correspondencia["scored"], readings))

## 6 · La masa en la clase verdadera

Cuánta de la masa de correspondencia de cada bolsa de destino cae sobre bolsas de
fuente de su propia clase, promediada sobre todas las bolsas en vez de sobre los
pocos sujetos que la figura de arriba destaca. Es la afirmación propia del término
local, dicha como número: la figura muestra unos sujetos, esto mide todos. Buscamos
que suba claramente por encima del azar, y que suba
más en el método que declara el término que en el mismo método sin él.

In [ ]:
show(tables.objective("correspondence.massOnTrueClass"))

In [ ]:
show(tables.render_readings(readings, "correspondence.massOnTrueClass",
                            f"masa en la clase verdadera "
                            f"(azar {1 / config.CLASSES:.3f})", markdown=True))

In [ ]:
show(tables.conclusion_mass(readings))

#### La masa en la clase verdadera, con el material contaminado

La misma lectura sobre la campaña a ρ del paquete. La conclusión de abajo informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_readings_contaminated(
    readings_ruido, "correspondence.massOnTrueClass", "La masa en la clase verdadera", RHO, markdown=True))


In [ ]:
show(tables.conclusion_readings_versus_clean(
    readings, readings_ruido, "correspondence.massOnTrueClass", RHO)
     if readings_ruido else
     "Sin la corrida contaminada no hay distancia que medir.")

### 6b · Cuánto reparte la atención

La entropía de los pesos que la atención asigna dentro de una bolsa. Dice si la
bolsa se apoya en unas pocas instancias o las pesa a todas casi por igual, que es lo
que hace falta para saber si la masa de arriba viene de una correspondencia real o
de una atención que dejó de elegir. Es descriptivo: se informa y no se disputa, pero
una atención plana vuelve discutible cualquier lectura de la tabla anterior.

In [ ]:
show(tables.objective("attentionSpread"))

In [ ]:
show(tables.render_readings(readings, "attentionSpread",
                            "entropía de los pesos dentro de la bolsa", markdown=True))

In [ ]:
show(tables.conclusion_attention(readings))

#### El reparto de la atención, con el material contaminado

La misma lectura sobre la campaña a ρ del paquete. La conclusión de abajo informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_readings_contaminated(
    readings_ruido, "attentionSpread", "El reparto de la atención", RHO, markdown=True))


In [ ]:
show(tables.conclusion_readings_versus_clean(
    readings, readings_ruido, "attentionSpread", RHO)
     if readings_ruido else
     "Sin la corrida contaminada no hay distancia que medir.")

## 7 · El registro

Escribe `latent.json` con cada lectura, cada comparación contra el piso y cada
conclusión recalculada, y `latent.md` con las mismas tablas de arriba en el mismo
orden. Una sesión posterior tiene que poder saber que esta corrida ocurrió, bajo qué
reducción y contra qué revisión, y nada de eso vive fuera del repositorio. Se genera
junto con los resultados y nunca se escribe a mano: un resumen escrito a mano es una
segunda fuente de verdad, se desactualiza en silencio y se le cree igual.

In [ ]:
# La comparación de cada método contra su propio piso ya no tiene sección en el
# cuaderno, y sigue siendo una medición que el registro tiene que llevar.
compared = latent.against_floor(readings)

output = DESTINO / "latent.json"
output.write_text(json.dumps({
    "revision": config.REVISION,
    "displaySeed": seed,
    "figureTransfers": transfers,
    "figureTransferRule": config.FIGURE_TRANSFER_RULE,
    "gridPanels": paneles,
    "gridUnit": config.LATENT_UNIT,
    "bagPanels": config.BAG_PANELS,
    "floorsAgree": {k: v for k, v in pisos.items() if k != "byTransfer"},
    "readings": readings,
    "againstFloor": compared,
    "correspondence": correspondencia["scored"],
    # Lo que midió la campaña, copiado tal cual en la misma corrida que lo leyó.
    # Sin esto el registro no alcanza para volver a producir sus propias
    # conclusiones, y una conclusión que no se puede volver a calcular no se puede
    # comprobar contra nada.
    "reduction": summary["reduction"],
    "panorama": summary["panorama"],
    "conclusions": tables.conclusions({
        "runs": runs, "reduction": summary["reduction"],
        "panorama": summary["panorama"], "readings": readings,
        "correspondence": correspondencia["scored"],
    }),
    "figures": [str(grid_path.relative_to(REPOSITORY)),
                str(correspondencia["path"].relative_to(REPOSITORY))],
}, indent=2), encoding="utf-8")

(DESTINO / "latent.md").write_text("\n\n".join([
    f"# Representación latente (v1) — {config.REVISION}",
    tables.stamp(summary["reduction"], markdown=True),
    "## 1 · Razón entre distancias (más bajo es mejor, si la de entre clases no cayó)",
    tables.render_readings(readings, "geometry.ratio",
                           "razón: misma clase entre dominios / entre clases",
                           markdown=True),
    tables.conclusion_geometry(readings),
    "### 1b · Las dos distancias por separado (descriptivas, se leen juntas)",
    tables.render_readings(readings, "geometry.crossDomainSameClass",
                           "misma clase, entre los dos dominios", markdown=True),
    tables.render_readings(readings, "geometry.betweenClasses",
                           "clases distintas, dentro de un dominio", markdown=True),
    tables.conclusion_distances(readings),
    f"## 2 · Separabilidad de dominio (más cerca de {tables.DOMAIN_CHANCE:.3f} es mejor)",
    tables.render_readings(readings, "domainSeparability",
                           "exactitud de un clasificador de dominio", markdown=True),
    tables.conclusion_separability(readings),
    f"## 3 · Masa en la clase verdadera (más alto es mejor, azar {1 / config.CLASSES:.3f})",
    tables.render_readings(readings, "correspondence.massOnTrueClass",
                           "masa en la clase verdadera", markdown=True),
    tables.conclusion_mass(readings),
    "### 3b · Cuánto reparte la atención (descriptivo)",
    tables.render_readings(readings, "attentionSpread",
                           "entropía de los pesos dentro de la bolsa", markdown=True),
    tables.conclusion_attention(readings),
    "## 4 · La correspondencia, sujeto por sujeto",
    tables.render_correspondence(correspondencia["scored"], markdown=True),
    tables.conclusion_correspondence(correspondencia["scored"], readings),
]), encoding="utf-8")
print("escrito", output.relative_to(REPOSITORY))

In [ ]:
# El sello: contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())